# Triple differences in ModernDiD and R

This notebook is generated from the [Triple differences in ModernDiD and R](https://moderndid.readthedocs.io/en/stable/r_comparisons/r_triple.html) page of the ModernDiD documentation.

This comparison checks whether the crop insurance analysis of the household
data from Cai (2016) gives the same results in ModernDiD and R's
`triplediff` package. After fitting [`ddd`](https://moderndid.readthedocs.io/en/stable/api/generated/didtriple/moderndid.ddd.html) in each language,
we inspect the effect for every year and compare the summaries from
[`agg_ddd`](https://moderndid.readthedocs.io/en/stable/api/generated/didtriple/moderndid.agg_ddd.html). The R calls run through `rpy2` on the same
observations used by Python.

Every post-treatment effect and all three summaries agree to numerical
precision. The doubly robust estimator of Ortiz-Villavicencio and Sant'Anna
(2025) conditions its propensity and outcome models on one set of
pre-treatment covariates for each household. Since `triplediff` 0.2.4 reads
those covariates from two different years in each pre-treatment comparison,
the effects for 2000 and 2001 differ by up to 1.7e-5. The [section on
pre-treatment covariates](https://moderndid.readthedocs.io/en/stable/r_comparisons/r_triple.html#covariates-in-the-pre-treatment-comparisons) traces
that gap to the R code.

> **What this comparison checks**
>
> Both fits adjust for household size and the age of the household head, use
> never-treated counties as controls, and anchor the effects to 2002. The
> comparison covers every yearly effect before and after treatment and the
> event-study, simple, and group summaries. The
> [triple-difference example](https://moderndid.readthedocs.io/en/stable/user_guide/example_triple_did.html#example-triple-did) also analyzes the
> unbalanced sample and bootstrap inference.

In [ ]:
import subprocess
import sys

if "google.colab" in sys.modules:
    # Colab ships rpy2 3.5 as a single package. Uninstalling it first keeps pip from deleting
    # the files it shares with the split rpy2 3.6 packages.
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "rpy2"], check=True)
    # rpy2 supports Colab's R 4.6 from release 3.6.7 on.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "moderndid[all]==0.4.1", "rpy2>=3.6.7"], check=True)
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "r-cran-triplediff"], check=True)
else:
    print("Outside Colab, set up R as https://moderndid.readthedocs.io/en/stable/r_comparisons/index.html#r-comparisons-running describes.")

## Use the same households and years

[`load_cai2016`](https://moderndid.readthedocs.io/en/stable/api/generated/data/moderndid.load_cai2016.html) contains households observed between 2000
and 2008. Insurance became available to tobacco farmers in the treated
county in 2003. The outcome, `checksaving_ratio`, is the share of net
savings held in checking accounts. `sector` marks tobacco farmers, the
households eligible for insurance. We use
[`get_group`](https://moderndid.readthedocs.io/en/stable/api/generated/panel/moderndid.core.panel.get_group.html) to turn the county's treatment flag
into an adoption year of 2003 for the treated county and zero elsewhere.

Since the original panel is unbalanced, this comparison keeps only
households observed in every year with a finite outcome and finite
covariates. This selection happens before the data reach either estimator
and keeps 3,298 of the 3,659 households and 29,682 of the 32,391
observations. The section on
[running the comparisons](https://moderndid.readthedocs.io/en/stable/r_comparisons/index.html#r-comparisons-running) explains how to rerun
the notebook.

In [ ]:
import polars as pl
import rpy2.robjects as ro

import moderndid as did

data = did.get_group(
    did.load_cai2016(), idname="hhno", tname="year", treatname="treatment", treat_period=2003
).rename({"G": "group"})
required = ["hhno", "year", "group", "sector", "checksaving_ratio", "hhsize", "age"]
data = data.filter(pl.all_horizontal(pl.col(required).is_finite()))
data = data.filter(pl.len().over("hhno") == 9)

In [ ]:
import numpy as np
import pandas as pd
from rpy2.robjects import pandas2ri

with (ro.default_converter + pandas2ri.converter).context():
    ro.globalenv["cai"] = ro.conversion.py2rpy(data.to_pandas())

# Eight decimals keep the small tolerance bounds in the pre-treatment table readable.
pd.set_option("display.precision", 8)

The triple difference compares four groups of households that differ by
county and by whether they farm tobacco. In the county that offered insurance
(`group=2003`), 765 households farm tobacco (`sector=1`) and 149 do not. In
the other counties (`group=0`), 1,150 households farm tobacco and 1,234 do not.

## Fit the yearly effects in Python

We use the doubly robust estimator with `~ hhsize + age` and a universal
base period. Both implementations receive the prepared balanced panel and
use analytical standard errors.

In [ ]:
py_fit = did.ddd(
    data=data,
    yname="checksaving_ratio",
    tname="year",
    idname="hhno",
    gname="group",
    pname="sector",
    xformla="~ hhsize + age",
    est_method="dr",
    control_group="nevertreated",
    base_period="universal",
    panel=True,
    allow_unbalanced_panel=False,
    boot=False,
)
print(py_fit)

## Fit the same effects in R

The R call names the same outcome, covariates, adoption year, and
eligibility partition. Its summary prints the results of `triplediff::ddd`
exactly as the package reports them. In its 2000 row the effect rounds to
−0.0565, against −0.0564 in the Python summary above.

In [ ]:
with ro.default_converter.context():
    ro.r("""
        r_fit <- triplediff::ddd(
            data = cai,
            yname = "checksaving_ratio",
            tname = "year",
            idname = "hhno",
            gname = "group",
            pname = "sector",
            xformla = ~ hhsize + age,
            est_method = "dr",
            control_group = "nevertreated",
            base_period = "universal",
            panel = TRUE,
            allow_unbalanced_panel = FALSE,
            boot = FALSE,
            cband = FALSE
        )
    """)
    r_fit = ro.globalenv["r_fit"]
    _ = ro.r("summary(r_fit)")

## Put each yearly result next to its counterpart

We join the outputs by treatment cohort and year and require every row to
appear in both fits. The difference columns report Python minus R in
scientific notation so that small numerical differences stay visible.

> **The normalized reference year**
>
> The universal base period anchors this cohort's effects to 2002, the year
> before insurance arrived. That row's ATT is fixed at zero with an undefined
> standard error. We keep the row visible and check standard errors only for
> the estimated effects.

In [ ]:
python_cells = pd.DataFrame({
    "cohort": py_fit.groups,
    "year": py_fit.times,
    "Python ATT": py_fit.att,
    "Python SE": py_fit.se,
})
with ro.default_converter.context():
    assert r_fit.rx2("n")[0] == py_fit.n == data["hhno"].n_unique()
    r_cells = pd.DataFrame({
        "cohort": r_fit.rx2("groups"),
        "year": r_fit.rx2("periods"),
        "R ATT": r_fit.rx2("ATT"),
        "R SE": r_fit.rx2("se"),
    })

comparison = python_cells.merge(r_cells, on=["cohort", "year"], validate="one_to_one")
assert len(comparison) == len(python_cells) == len(r_cells)
comparison["ATT difference"] = comparison["Python ATT"] - comparison["R ATT"]
comparison["SE difference"] = comparison["Python SE"] - comparison["R SE"]
comparison

### Covariates in the pre-treatment comparisons

You can see in the table that the post-treatment rows differ by at most
1.5e-14 in the estimates and 1.1e-12 in the standard errors. The 2000 and
2001 rows differ by 1.7e-5 and 6.0e-6 in the ATT and by 2.9e-6 and 1.2e-6 in
the standard error. That is enough to round the 2000 effect to −0.0564 in
ModernDiD and −0.0565 in R.

With a universal base period, every outcome change is measured from 2002.
ModernDiD takes household size and age from 2002 for the propensity model,
the outcome model, and the influence function in every comparison.
`triplediff` 0.2.4 reads the outcome model's covariates from 2002 too. Its
propensity model and the outcome-model terms of its influence function
instead read each household's first row. After treatment starts, that row is
from 2002 as well. In a pre-treatment comparison, it comes from the earlier
year, 2000 or 2001.

> **Why covariates come from the base year**
>
> ModernDiD conditions each comparison on covariates from its base period, the
> year that the outcome change is measured from. Since that year always
> precedes treatment, the covariates cannot respond to it. The propensity
> model, the outcome model, and the influence function then share one set of
> covariates.

In the R source, `compute_pscore` and `compute_did` take that first row with
`unique(..., by = "id")` (`R/compute_nuisances.R`, lines 55 and 203). The
outcome regression conditions on the 2002 rows instead (lines 145 to 148 and
173 to 176). A comment there justifies this by assuming that covariates
never change. The earlier-year propensity covariates move R's estimate for
those two years. Whenever covariates change over time, R's standard error
also comes from an influence function that linearizes a regression fitted on
different covariates. The two-period path of `triplediff` stops with
"Covariates are not time invariant" in that case (`R/preprocess.R`, lines
540 to 548). The multi-period path used here runs without any such check.

The gap stays tiny on this data because between 2000 and 2002 the head's age
rises by two years in all but 28 of the 3,298 households and household size
changes in only two. Since a shift that every household shares leaves the
fitted models unchanged, only those few households move the estimate.

## Compare the event study

Event-study aggregation counts each year from the 2003 rollout and places
2002 at event time −1. With one adoption cohort, these effects correspond
directly to the yearly effects above. Event times −3 and −2 therefore carry
the same small gaps as 2000 and 2001.

In [ ]:
py_dynamic = did.agg_ddd(py_fit, type="eventstudy", boot=False, cband=False)
print(py_dynamic)

In [ ]:
with ro.default_converter.context():
    ro.r('r_dynamic <- triplediff::agg_ddd(r_fit, type = "eventstudy", boot = FALSE, cband = FALSE)')
    r_dynamic = ro.globalenv["r_dynamic"]
    _ = ro.r("summary(r_dynamic)")

In [ ]:
python_events = pd.DataFrame({
    "event time": py_dynamic.egt,
    "Python ATT": py_dynamic.att_egt,
    "Python SE": py_dynamic.se_egt,
})
with ro.default_converter.context():
    r_dynamic_values = r_dynamic.rx2("aggte_ddd")
    r_events = pd.DataFrame({
        "event time": r_dynamic_values.rx2("egt"),
        "R ATT": r_dynamic_values.rx2("att.egt"),
        "R SE": r_dynamic_values.rx2("se.egt"),
    })

event_comparison = python_events.merge(r_events, on="event time", validate="one_to_one")
assert len(event_comparison) == len(python_events) == len(r_events)
event_comparison["ATT difference"] = event_comparison["Python ATT"] - event_comparison["R ATT"]
event_comparison["SE difference"] = event_comparison["Python SE"] - event_comparison["R SE"]
event_comparison

## Compare the overall summaries

We also fit simple and group aggregation in both languages. Simple
aggregation weights the post-treatment cohort-year effects by the size of
each cohort. Group aggregation first averages within each treated cohort and
then across cohorts. After the native summaries, one table puts the overall
ATT and its standard error side by side for all three choices.

In [ ]:
py_simple = did.agg_ddd(py_fit, type="simple", boot=False, cband=False)
print(py_simple)

In [ ]:
with ro.default_converter.context():
    ro.r('r_simple <- triplediff::agg_ddd(r_fit, type = "simple", boot = FALSE, cband = FALSE)')
    r_simple = ro.globalenv["r_simple"]
    _ = ro.r("summary(r_simple)")

In [ ]:
py_group = did.agg_ddd(py_fit, type="group", boot=False, cband=False)
print(py_group)

In [ ]:
with ro.default_converter.context():
    ro.r('r_group <- triplediff::agg_ddd(r_fit, type = "group", boot = FALSE, cband = FALSE)')
    r_group = ro.globalenv["r_group"]
    _ = ro.r("summary(r_group)")

In [ ]:
overall_rows = []
for kind, py_aggregate, r_aggregate in (
    ("simple", py_simple, r_simple),
    ("group", py_group, r_group),
    ("eventstudy", py_dynamic, r_dynamic),
):
    with ro.default_converter.context():
        r_values = r_aggregate.rx2("aggte_ddd")
        r_att = r_values.rx2("overall.att")[0]
        r_se = r_values.rx2("overall.se")[0]
    overall_rows.append({
        "aggregation": kind,
        "Python ATT": py_aggregate.overall_att,
        "R ATT": r_att,
        "ATT difference": py_aggregate.overall_att - r_att,
        "Python SE": py_aggregate.overall_se,
        "R SE": r_se,
        "SE difference": py_aggregate.overall_se - r_se,
    })
overall_comparison = pd.DataFrame(overall_rows)
overall_comparison

Since this data has one insured cohort, all three summaries average the same
six post-treatment years. Their overall estimates therefore answer the same
question here and leave out the pre-treatment years where the two packages
part. With a single cohort, this comparison cannot test how the aggregations
weight several adoption cohorts.

That limit hides a difference in how the two packages measure a cohort's size.
`triplediff` computes each cohort's share from every household in its enabling
cohort, eligible or not (`R/compute_aggregations.R`, line 346). Its term for
the estimated shares uses the same cohort indicator (line 356). Following the
cohort definition of Ortiz-Villavicencio and Sant'Anna (2025), ModernDiD
counts only the eligible households that receive treatment. With several
cohorts, the standard errors of summaries that average them therefore differ.
The estimates differ too when the cohorts hold different shares of eligible
households. The [timing runner](https://moderndid.readthedocs.io/en/stable/dev/benchmarking.html#benchmarks-numerical-checks) records
these standard-error gaps on its generated panels.

The group summary's standard error has a second source of difference.
`triplediff` passes the cohort of every group-time cell to
`get_weight_influence` where the list of distinct cohorts belongs (lines 478
to 482). That function reads the k-th entry as the k-th cohort (lines 33 to
41). Since the cells are sorted by cohort, the first entries all repeat the
earliest cohort once there are two or more. The term for estimated cohort
shares then pairs each later cohort's share with the earliest cohort's
indicator. ModernDiD builds that term from the list of distinct cohorts
instead.

> **R's group standard error reuses the earliest cohort**
>
> Although the group estimate stays the same, the pairing alone left R's group
> standard error about 10 percent below the one from the distinct cohorts on a
> simulated panel with three cohorts whose effects differ. Report ModernDiD's
> group standard error, since its share term pairs each cohort's share with that
> cohort's own indicator.

Because the only insured cohort takes all the weight in each summary, the
share term vanishes on the crop insurance data. Both packages therefore report
a standard error of 0.0157 for every summary.

## Check the numerical differences

Each ATT may differ from R by at most `1e-6 + 1e-5 * abs(R estimate)`. Each
standard error may differ by at most `1e-6 + 1e-3 * abs(R SE)`. The next
table applies those bounds to the two pre-treatment contrasts. Their ATT
gaps exceed the estimate bound because of the [covariate
years](https://moderndid.readthedocs.io/en/stable/r_comparisons/r_triple.html#covariates-in-the-pre-treatment-comparisons) that the two packages
use. Their standard-error gaps stay well inside the standard-error bound.

In [ ]:
pre_treatment = comparison[comparison["year"] < 2002].copy()
pre_treatment["ATT bound"] = 1e-6 + 1e-5 * pre_treatment["R ATT"].abs()
pre_treatment["SE bound"] = 1e-6 + 1e-3 * pre_treatment["R SE"].abs()
pre_treatment[["year", "ATT difference", "ATT bound", "SE difference", "SE bound"]]

Checks against those bounds cover every post-treatment yearly and event-time
effect and all three overall summaries. They leave out the two pre-treatment
contrasts because their gaps come from the covariate split. The normalized
reference year is checked separately because its standard error is
undefined. The final table reports the largest difference in each checked
comparison.

In [ ]:
post_treatment = comparison[comparison["year"] >= 2003]
post_treatment_events = event_comparison[event_comparison["event time"] >= 0]
for table in (post_treatment, post_treatment_events, overall_comparison):
    assert np.isfinite(table[["Python ATT", "R ATT", "Python SE", "R SE"]]).all().all()
    np.testing.assert_allclose(table["Python ATT"], table["R ATT"], rtol=1e-5, atol=1e-6)
    np.testing.assert_allclose(table["Python SE"], table["R SE"], rtol=1e-3, atol=1e-6)
reference = comparison[comparison["year"] == 2002]
np.testing.assert_array_equal(reference["Python ATT"], reference["R ATT"])

pd.DataFrame([
    {
        "comparison": label,
        "largest absolute ATT difference": table["ATT difference"].abs().max(),
        "largest absolute SE difference": table["SE difference"].abs().max(),
    }
    for label, table in (
        ("post-treatment effects", post_treatment),
        ("post-treatment event times", post_treatment_events),
        ("overall summaries", overall_comparison),
    )
])

## The unbalanced sample

The [triple-difference example](https://moderndid.readthedocs.io/en/stable/user_guide/example_triple_did.html#example-triple-did) keeps all 3,659
households with `allow_unbalanced_panel=True`. Each comparison then pools the
households observed in either of its two years and uses the repeated
cross-section estimator. Although the standard errors on that sample differ by
design, every estimate matches `triplediff`.

In each of the three comparisons, the pre-treatment averages are subtracted
from the post-treatment ones. Those averages use a regression fitted to the
comparison households' pre-treatment outcomes. Its estimation effect therefore
enters the influence function with a minus sign as well. `triplediff` 0.2.4
adds it instead (`R/compute_nuisances.R`, lines 729 and 730). Line 731 subtracts
the matching term of the locally efficient adjustment. A finite-difference
derivative of the estimate confirms the minus sign for all three terms. With
household size and age as covariates, the analytic standard errors of
`triplediff` run from 6 percent below ModernDiD's to 4 percent above. The
reference tests in `tests/validation/test_r_didtriple.py` match `triplediff`
once they correct that sign inside R.

## Versions used

These versions identify the software that produced the stored output above.
Rerunning the notebook records the installed versions with the new results.

In [ ]:
from importlib.metadata import version
from platform import python_version

with ro.default_converter.context():
    r_version = ro.r("R.version.string")[0]
    triplediff_version = ro.r('as.character(packageVersion("triplediff"))')[0]
pd.DataFrame({
    "component": ["Python", "ModernDiD", "rpy2", "R", "triplediff"],
    "version": [python_version(), version("moderndid"), version("rpy2"), r_version, triplediff_version],
})